
# 03 — Resultados de Scoring RADAR Cibest | Versión Deep Dive Interpretativa

**Fase ASUM-DM:** 4 — Modelado y lectura estratégica de resultados  
**Proyecto:** RADAR Cibest — Ranking de Atractivo y Diagnóstico Analítico Regional  
**Propósito:** convertir los resultados de TOPSIS, IPC, Trend y RADAR en una lectura ejecutiva orientada a decisión, conectando rankings, tesis de negocio, drivers, restricciones, atractivo común entre países, sensibilidad por línea y consecuencias estratégicas.

---

## Tesis central del notebook

RADAR no debe leerse como una tabla de posiciones. Debe leerse como un sistema de interpretación estratégica:

1. **TOPSIS** identifica atractivo estructural relativo: qué países se parecen más al ideal positivo definido por fundamentos macro, financieros, institucionales y digitales.
2. **IPC** mide afinidad específica con Colombia: qué mercados son más accesibles para Cibest por cercanía geográfica, cultural, migratoria y relacional.
3. **Trend** captura momentum macroeconómico reciente: qué países están mejorando en el margen.
4. **RADAR** integra los tres componentes para priorizar mercados bajo una tesis de internacionalización realista: atractivo absoluto + proximidad ejecutable + momentum.
5. **Rankings por línea** no son listas independientes: son variaciones de una misma matriz país, ponderada según la lógica económica de cada negocio.

La pregunta ejecutiva no es solo: **¿quién queda de primero?**  
La pregunta correcta es: **¿por qué queda arriba, qué tan separada es su ventaja, para qué línea de negocio es atractivo, qué drivers lo explican y qué riesgos de implementación persisten?**



## 1. Configuración reproducible y recarga controlada de módulos

Esta celda evita uno de los problemas más frecuentes del proyecto: ejecutar notebooks con módulos antiguos en memoria después de modificar `src/`. La recarga debe ocurrir **antes** de importar funciones sueltas.


In [388]:

# ---------------------------------------------------------------------------
# Configuración inicial, imports y recarga controlada de módulos
# ---------------------------------------------------------------------------
from __future__ import annotations

import sys
import re
import importlib
import inspect
import warnings
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import HTML, display, Markdown

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

import src.utils as utils
import src.data_preparation.cleaning as cleaning
import src.data_preparation.feature_engineering as feature_engineering
import src.scoring.hybrid_scorer as hybrid_scorer
import src.scoring.ranking as ranking
import src.scoring.explainability as explainability
import src.scoring.monte_carlo as monte_carlo

importlib.invalidate_caches()

# Primero dependencias; luego módulos que importan esas dependencias.
importlib.reload(utils)
importlib.reload(cleaning)
importlib.reload(feature_engineering)
importlib.reload(ranking)
importlib.reload(explainability)
importlib.reload(monte_carlo)
importlib.reload(hybrid_scorer)

from src.utils import load_all_configs, setup_logger, resolve_data_path, get_variable_catalog
from src.data_preparation.cleaning import run_cleaning
from src.scoring.hybrid_scorer import prepare_decision_matrix, run_full_scoring, _build_business_line_weights
from src.scoring.explainability import (
    compute_all_business_line_contributions,
    compute_all_marginal_effects,
    combine_contribution_and_marginal,
    classify_driver_robustness,
    build_country_driver_table,
)

configs = load_all_configs()
setup_logger(configs["settings"].get("logging"))
variable_catalog = get_variable_catalog(configs["variables"])

print("cleaning file:", cleaning.__file__)
print("hybrid_scorer file:", hybrid_scorer.__file__)
print("run_cleaning line:", inspect.getsourcelines(cleaning.run_cleaning)[1])
print("prepare_decision_matrix line:", inspect.getsourcelines(hybrid_scorer.prepare_decision_matrix)[1])


cleaning file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\data_preparation\cleaning.py
hybrid_scorer file: c:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\src\scoring\hybrid_scorer.py
run_cleaning line: 355
prepare_decision_matrix line: 37



## 2. Estilo visual Cibest y utilidades ejecutivas

Las visualizaciones privilegian lectura ejecutiva: poco ornamento, colores con significado, tablas con gradientes solo cuando ayudan a identificar patrones.


In [389]:

# ---------------------------------------------------------------------------
# Paleta Cibest y utilidades de visualización
# ---------------------------------------------------------------------------
CIBEST: Dict[str, str] = {
    "gray": "#2C2A28",
    "gray_light": "#CCCAC7",
    "yellow": "#FDD923",
    "gold": "#D6B302",
    "gold_light": "#FFF7D3",
    "gold_dark": "#8F7701",
    "gray_bg": "#F5F5F5",
    "gray_border": "#D0D0D0",
    "white": "#FFFFFF",
    "green": "#0BA682",
    "amber": "#FF7E41",
    "red": "#C62828",
}

TIER_COLORS: Dict[str, str] = {
    "Alta": CIBEST["green"],
    "Media-alta": CIBEST["gold"],
    "Media": CIBEST["amber"],
    "Baja": CIBEST["red"],
}


cmap_custom = LinearSegmentedColormap.from_list(
    "GrayToYellow",
    [CIBEST["gray_light"], CIBEST["yellow"]],
)

def style_table(
    df: pd.DataFrame,
    gradient_cols: Optional[List[str]] = None,
    gradient_cmap: str = "YlGnBu",
    format_dict: Optional[Dict[str, str]] = None,
):
    """Aplica estilo visual Cibest a un DataFrame."""
    styler = df.style.set_table_styles([
        {"selector": "th", "props": [
            ("background-color", CIBEST["gray"]),
            ("color", CIBEST["yellow"]),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "8px"),
            ("font-family", "Arial, sans-serif"),
        ]},
        {"selector": "td", "props": [
            ("padding", "6px 10px"),
            ("font-family", "Arial, sans-serif"),
            ("border-bottom", f"1px solid {CIBEST['gray_border']}"),
        ]},
    ])
    if gradient_cols:
        cols = [c for c in gradient_cols if c in df.columns]
        if cols:
            styler = styler.background_gradient(subset=cols, cmap=gradient_cmap)
    if format_dict:
        styler = styler.format(format_dict)
    return styler


def insight_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de insight."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['gold']}; background-color:{CIBEST['gold_light']};
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def risk_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de riesgo."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['red']}; background-color:#FDECEC;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def success_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de confirmación."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['green']}; background-color:#E8F7F3;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))

success_box("Entorno listo", "Estilo ejecutivo y módulos RADAR recargados correctamente.")



## 3. Carga del master y validación mínima

La lectura de resultados parte del `master_raw_YYYYMMDD.parquet` más reciente. Esta celda no interpreta todavía; valida que el insumo del scoring sea completo y trazable.


In [390]:

# ---------------------------------------------------------------------------
# Carga del master raw más reciente
# ---------------------------------------------------------------------------
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])
pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

candidates = sorted(
    [path for path in raw_dir.glob("master_raw_*.parquet") if pattern.match(path.name)],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not candidates:
    raise FileNotFoundError("No se encontró master_raw_YYYYMMDD.parquet. Ejecute primero el notebook 01.")

master_path = candidates[0]
master = pd.read_parquet(master_path)

required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)
if missing_cols:
    raise ValueError(f"Master inválido. Faltan columnas requeridas: {sorted(missing_cols)}")

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["value"] = pd.to_numeric(master["value"], errors="coerce")

summary_master = pd.DataFrame({
    "métrica": ["archivo", "filas", "países", "variables", "fuentes", "año_min", "año_max", "incluye_gdp_growth"],
    "valor": [
        master_path.name,
        master.shape[0],
        master["country_iso3"].nunique(),
        master["variable"].nunique(),
        master["source"].nunique(),
        int(master["year"].min()),
        int(master["year"].max()),
        "Sí" if "gdp_growth" in master["variable"].unique() else "No",
    ],
})

display(style_table(summary_master))

if "gdp_growth" not in master["variable"].unique():
    risk_box("Advertencia crítica", "El master no contiene gdp_growth. El componente Trend no podrá calcularse correctamente.")


,métrica,valor
0,archivo,master_raw_20260626.parquet
1,filas,1318
2,países,30
3,variables,46
4,fuentes,3
5,año_min,1996
6,año_max,2026
7,incluye_gdp_growth,Sí



## 4. Preparación de matriz y política de exclusión por calidad

La matriz de scoring debe construirse con una regla conservadora: **primero se excluyen países con faltantes efectivos excesivos; luego se imputa solo lo residual**. Esto es relevante porque TOPSIS usa distancias al ideal positivo y negativo; países con demasiada imputación pueden alterar esos puntos de referencia.


In [391]:

# ---------------------------------------------------------------------------
# Preparación de matriz de decisión y exclusiones
# ---------------------------------------------------------------------------
wide_raw, decision_matrix, excluded_countries = prepare_decision_matrix(master, configs)

matrix_summary = pd.DataFrame({
    "métrica": [
        "wide_raw_shape",
        "decision_matrix_shape",
        "países_excluidos",
        "gdp_growth_en_wide_raw",
        "gdp_growth_en_decision_matrix",
    ],
    "valor": [
        str(wide_raw.shape),
        str(decision_matrix.shape),
        ", ".join(sorted(excluded_countries)) if excluded_countries else "Ninguno",
        "Sí" if "gdp_growth" in wide_raw.columns else "No",
        "Sí" if "gdp_growth" in decision_matrix.columns else "No",
    ],
})

display(style_table(matrix_summary))

if "gdp_growth" in decision_matrix.columns:
    raise ValueError("gdp_growth está entrando a TOPSIS. Debe usarse solo en Trend.")

if excluded_countries:
    risk_box(
        "Exclusiones por calidad antes de scoring",
        "Los países excluidos no entran a TOPSIS/RADAR. Esto evita que la imputación convierta países con baja cobertura en alternativas artificialmente comparables. "
        f"Excluidos: {', '.join(sorted(excluded_countries))}."
    )
else:
    success_box("Sin exclusiones por calidad", "Todos los países cumplen el umbral de cobertura efectiva definido en configuración.")


2026-06-26 00:55:24 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 46 variables (estrategia=latest_available)


--- Logging error in Loguru Handler #26 ---
Record was: {'elapsed': datetime.timedelta(days=2, seconds=42006, microseconds=945167), 'exception': None, 'extra': {'c': 30, 'v': 46}, 'file': (name='cleaning.py', path='c:\\Users\\jadarve\\OneDrive - Grupo Bancolombia\\Bancolombia\\GFEC-VEF\\2026\\Internacionalización\\radar_cibest_v2\\src\\data_preparation\\cleaning.py'), 'function': 'pivot_latest_value_and_year', 'level': (name='INFO', no=20, icon='ℹ️'), 'line': 133, 'message': 'Pivoteo ancho: 30 paises x 46 variables (estrategia=latest_available)', 'module': 'cleaning', 'name': 'src.data_preparation.cleaning', 'process': (id=24300, name='MainProcess'), 'thread': (id=37796, name='MainThread'), 'time': datetime(2026, 6, 26, 0, 55, 24, 621129, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Hora est. Pacífico, Sudamérica'))}
Traceback (most recent call last):
  File "C:\Users\jadarve\AppData\Roaming\Python\Python39\site-packages\loguru\_handler.py", line 206, in emit
 

,métrica,valor
0,wide_raw_shape,"(25, 47)"
1,decision_matrix_shape,"(25, 36)"
2,países_excluidos,"BRB, CUB, GUY, HTI, VEN"
3,gdp_growth_en_wide_raw,Sí
4,gdp_growth_en_decision_matrix,No



## 5. Ejecución del scoring híbrido RADAR

La ejecución produce tres familias de resultados:

- **Global:** ranking estructural TOPSIS y ranking compuesto RADAR.
- **Componentes:** IPC y Trend, que explican movimientos respecto a TOPSIS.
- **Líneas de negocio:** rankings por IB, PF, AD, BD y CIB, cada uno con una tesis distinta de atractividad.


In [392]:

# ---------------------------------------------------------------------------
# Ejecución completa del scoring
# ---------------------------------------------------------------------------
results = run_full_scoring(master, configs, persist=True)

alpha = results["composite_weights"]["alpha"]
beta = results["composite_weights"]["beta"]
gamma = results["composite_weights"]["gamma"]

scoring_summary = pd.DataFrame({
    "métrica": ["alpha_TOPSIS", "beta_IPC", "gamma_Trend", "países_evaluados", "países_excluidos"],
    "valor": [
        alpha,
        beta,
        gamma,
        len(results["radar_global"]),
        ", ".join(sorted(results.get("excluded_countries", []))) if results.get("excluded_countries") else "Ninguno",
    ],
})

display(style_table(scoring_summary, format_dict={"valor": "{}"}))

insight_box(
    "Lectura del score compuesto",
    f"RADAR pondera {alpha:.0%} atractivo estructural TOPSIS, {beta:.0%} proximidad/afinidad con Colombia y {gamma:.0%} momentum macro reciente. "
    "Por diseño, el ranking final no replica exactamente el ranking estructural: mercados cercanos o con fuerte tendencia pueden subir, mientras mercados estructuralmente fuertes pero lejanos pueden bajar."
)


2026-06-26 00:55:25 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 46 variables (estrategia=latest_available)
--- Logging error in Loguru Handler #26 ---
Record was: {'elapsed': datetime.timedelta(days=2, seconds=42007, microseconds=349941), 'exception': None, 'extra': {'c': 30, 'v': 46}, 'file': (name='cleaning.py', path='c:\\Users\\jadarve\\OneDrive - Grupo Bancolombia\\Bancolombia\\GFEC-VEF\\2026\\Internacionalización\\radar_cibest_v2\\src\\data_preparation\\cleaning.py'), 'function': 'pivot_latest_value_and_year', 'level': (name='INFO', no=20, icon='ℹ️'), 'line': 133, 'message': 'Pivoteo ancho: 30 paises x 46 variables (estrategia=latest_available)', 'module': 'cleaning', 'name': 'src.data_preparation.cleaning', 'process': (id=24300, name='MainProcess'), 'thread': (id=37796, name='MainThread'), 'time': datetime(2026, 6, 26, 0, 55, 25, 25903, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Hora est. Pacíf

,métrica,valor
0,alpha_TOPSIS,0.6
1,beta_IPC,0.3
2,gamma_Trend,0.1
3,países_evaluados,24
4,países_excluidos,"BRB, CUB, GUY, HTI, VEN"



## 6. Ranking RADAR global: lectura ejecutiva del portafolio país

El ranking RADAR global debe interpretarse como una priorización inicial de mercados para discusión ejecutiva. No sustituye due diligence regulatoria, competitiva o financiera; identifica dónde tiene sentido profundizar.


In [393]:

# ---------------------------------------------------------------------------
# Ranking RADAR global y TOPSIS global
# ---------------------------------------------------------------------------
def ensure_country_column(df: pd.DataFrame) -> pd.DataFrame:
    """Asegura que country_iso3 exista como columna."""
    tmp = df.copy()
    if "country_iso3" not in tmp.columns:
        tmp = tmp.reset_index().rename(columns={"index": "country_iso3"})
    return tmp

radar_global = ensure_country_column(results["radar_global"]).sort_values("radar_score", ascending=False)
radar_global["radar_rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

global_topsis = ensure_country_column(results["global_ranking"]).sort_values("rank")

display(style_table(
    radar_global.head(15),
    gradient_cols=["radar_score"],
    gradient_cmap="RdYlGn",
    format_dict={"radar_score": "{:.3f}", "radar_rank": "{:.0f}"},
).set_caption("Top 15 — Ranking RADAR global"))

fig = px.bar(
    radar_global.head(15).sort_values("radar_score"),
    x="radar_score",
    y="country_iso3",
    orientation="h",
    color="radar_score",
    color_continuous_scale="RdYlGn",
    title="Top 15 mercados por score RADAR global",
)
fig.update_layout(xaxis_title="Score RADAR", yaxis_title="País")
fig.show()


,country_iso3,radar_score,rank,radar_rank
0,PAN,0.686,1,1
1,CRI,0.641,2,2
2,DOM,0.629,3,3
3,ESP,0.611,4,4
4,CHL,0.576,5,5
5,USA,0.554,6,6
6,URY,0.542,7,7
7,MEX,0.535,8,8
8,GTM,0.532,9,9
9,PER,0.520,10,10


In [394]:

# ---------------------------------------------------------------------------
# Interpretación automática del Top RADAR
# ---------------------------------------------------------------------------
top_radar = radar_global.head(5)[["country_iso3", "radar_score", "radar_rank"]].copy()
leader = top_radar.iloc[0]

interpretation = f"""
### Lectura ejecutiva del Top RADAR

El país líder del ranking compuesto es **{leader['country_iso3']}**, con un score RADAR de **{leader['radar_score']:.3f}**.  
El Top 5 está compuesto por: **{', '.join(top_radar['country_iso3'].tolist())}**.

La lectura correcta no es que el primer país sea "universalmente mejor", sino que, bajo la combinación actual de atractivo estructural, proximidad con Colombia y momentum macro, presenta la mejor mezcla relativa. Si el gap frente al segundo país es bajo, la decisión debe comunicarse como una banda competitiva y no como una jerarquía estricta.
"""

display(Markdown(interpretation))



### Lectura ejecutiva del Top RADAR

El país líder del ranking compuesto es **PAN**, con un score RADAR de **0.686**.  
El Top 5 está compuesto por: **PAN, CRI, DOM, ESP, CHL**.

La lectura correcta no es que el primer país sea "universalmente mejor", sino que, bajo la combinación actual de atractivo estructural, proximidad con Colombia y momentum macro, presenta la mejor mezcla relativa. Si el gap frente al segundo país es bajo, la decisión debe comunicarse como una banda competitiva y no como una jerarquía estricta.



## 7. De TOPSIS a RADAR: quién sube, quién baja y por qué

Esta es una de las lecturas más importantes del notebook. TOPSIS responde **“qué país es estructuralmente atractivo”**; RADAR responde **“qué país es atractivo para Cibest, considerando además proximidad y momentum”**.


In [395]:

# ---------------------------------------------------------------------------
# Descomposición RADAR y movimientos frente a TOPSIS
# ---------------------------------------------------------------------------
ipc_df = ensure_country_column(results["ipc"])
trend_df = ensure_country_column(results["trend"])

topsis_series = global_topsis.set_index("country_iso3")["score"].rename("topsis_score")
ipc_series = ipc_df.set_index("country_iso3")["ipc"].rename("ipc")
trend_series = trend_df.set_index("country_iso3")["trend"].rename("trend")

component_df = pd.concat([topsis_series, ipc_series, trend_series], axis=1)
component_df["ipc"] = component_df["ipc"].fillna(component_df["ipc"].median())
component_df["trend"] = component_df["trend"].fillna(component_df["trend"].median())
component_df["aporte_topsis"] = alpha * component_df["topsis_score"]
component_df["aporte_ipc"] = beta * component_df["ipc"]
component_df["aporte_trend"] = gamma * component_df["trend"]
component_df["radar_score_calc"] = component_df[["aporte_topsis", "aporte_ipc", "aporte_trend"]].sum(axis=1)
component_df["rank_topsis"] = component_df["topsis_score"].rank(ascending=False, method="min").astype(int)
component_df["rank_radar"] = component_df["radar_score_calc"].rank(ascending=False, method="min").astype(int)
component_df["delta_rank_radar_vs_topsis"] = component_df["rank_topsis"] - component_df["rank_radar"]
component_display = component_df.sort_values("rank_radar").reset_index().rename(columns={"index": "country_iso3"})

display(style_table(
    component_display.head(20),
    gradient_cols=["topsis_score", "ipc", "trend", "radar_score_calc", "delta_rank_radar_vs_topsis"],
    gradient_cmap="RdYlGn",
    format_dict={
        "topsis_score": "{:.3f}",
        "ipc": "{:.3f}",
        "trend": "{:.3f}",
        "aporte_topsis": "{:.3f}",
        "aporte_ipc": "{:.3f}",
        "aporte_trend": "{:.3f}",
        "radar_score_calc": "{:.3f}",
        "rank_topsis": "{:.0f}",
        "rank_radar": "{:.0f}",
        "delta_rank_radar_vs_topsis": "{:+.0f}",
    },
).set_caption("Descomposición RADAR y movimiento frente a TOPSIS"))


,country_iso3,topsis_score,ipc,trend,aporte_topsis,aporte_ipc,aporte_trend,radar_score_calc,rank_topsis,rank_radar,delta_rank_radar_vs_topsis
0,PAN,0.506,0.942,1.000,0.304,0.283,0.100,0.686,10,1,+9
1,CRI,0.524,0.835,0.763,0.315,0.250,0.076,0.641,7,2,+5
2,DOM,0.512,0.864,0.629,0.307,0.259,0.063,0.629,9,3,+6
3,ESP,0.617,0.596,0.621,0.370,0.179,0.062,0.611,4,4,+0
4,CHL,0.620,0.668,0.037,0.372,0.200,0.004,0.576,3,5,-2
5,USA,0.707,0.341,0.278,0.424,0.102,0.028,0.554,2,6,-4
6,URY,0.585,0.540,0.290,0.351,0.162,0.029,0.542,5,7,-2
7,MEX,0.491,0.703,0.303,0.294,0.211,0.030,0.535,11,8,+3
8,GTM,0.429,0.735,0.544,0.257,0.221,0.054,0.532,17,9,+8
9,PER,0.467,0.777,0.069,0.280,0.233,0.007,0.520,14,10,+4


In [396]:

# ---------------------------------------------------------------------------
# Lectura automática de ganadores y perdedores por IPC/Trend
# ---------------------------------------------------------------------------
raisers = component_display.sort_values("delta_rank_radar_vs_topsis", ascending=False).head(5)
fallers = component_display.sort_values("delta_rank_radar_vs_topsis", ascending=True).head(5)

text = f"""
### Movimientos estratégicos entre TOPSIS y RADAR

**Países que más suben al pasar de TOPSIS a RADAR:** {', '.join(raisers['country_iso3'].tolist())}.  
Estos mercados suelen beneficiarse de mayor IPC, mejor momentum o ambos. La implicación es que pueden no ser los más fuertes estructuralmente, pero sí ser más ejecutables para Cibest.

**Países que más bajan al pasar de TOPSIS a RADAR:** {', '.join(fallers['country_iso3'].tolist())}.  
Estos mercados suelen ser estructuralmente sólidos, pero pierden atractivo relativo cuando se incorpora cercanía con Colombia y tendencia reciente.

Esta comparación evita una lectura ingenua del ranking: un mercado puede ser excelente en abstracto, pero menos prioritario para Cibest si su distancia operativa o relacional es alta.
"""
display(Markdown(text))



### Movimientos estratégicos entre TOPSIS y RADAR

**Países que más suben al pasar de TOPSIS a RADAR:** PAN, ECU, GTM, NIC, DOM.  
Estos mercados suelen beneficiarse de mayor IPC, mejor momentum o ambos. La implicación es que pueden no ser los más fuertes estructuralmente, pero sí ser más ejecutables para Cibest.

**Países que más bajan al pasar de TOPSIS a RADAR:** BRA, CAN, JAM, BHS, TTO.  
Estos mercados suelen ser estructuralmente sólidos, pero pierden atractivo relativo cuando se incorpora cercanía con Colombia y tendencia reciente.

Esta comparación evita una lectura ingenua del ranking: un mercado puede ser excelente en abstracto, pero menos prioritario para Cibest si su distancia operativa o relacional es alta.



## 8. Arquetipos estratégicos de países

Para pasar de ranking a decisión, clasificamos países según la combinación de estructura, proximidad y momentum. Esta clasificación ayuda a definir **qué tipo de conversación estratégica** corresponde a cada país.


In [397]:

# ---------------------------------------------------------------------------
# Arquetipos estratégicos por componentes
# ---------------------------------------------------------------------------
def classify_archetype(row: pd.Series) -> str:
    """Clasifica un país según su combinación relativa de TOPSIS, IPC y Trend."""
    high_topsis = row["topsis_score"] >= component_df["topsis_score"].quantile(0.70)
    high_ipc = row["ipc"] >= component_df["ipc"].quantile(0.70)
    high_trend = row["trend"] >= component_df["trend"].quantile(0.70)

    if high_topsis and high_ipc:
        return "Prioridad natural Cibest"
    if high_topsis and not high_ipc:
        return "Atractivo estructural lejano"
    if high_ipc and not high_topsis:
        return "Cercano pero requiere tesis selectiva"
    if high_trend and not high_topsis:
        return "Momentum táctico"
    return "Monitoreo / oportunidad condicionada"

archetypes = component_df.copy()
archetypes["archetype"] = archetypes.apply(classify_archetype, axis=1)
archetype_display = (
    archetypes.reset_index()
    .rename(columns={"index": "country_iso3"})
    .sort_values("radar_score_calc", ascending=False)
)

display(style_table(
    archetype_display[["country_iso3", "archetype", "topsis_score", "ipc", "trend", "radar_score_calc", "rank_radar"]].head(25),
    gradient_cols=["topsis_score", "ipc", "trend", "radar_score_calc"],
    gradient_cmap="RdYlGn",
    format_dict={"topsis_score": "{:.3f}", "ipc": "{:.3f}", "trend": "{:.3f}", "radar_score_calc": "{:.3f}", "rank_radar": "{:.0f}"},
).set_caption("Arquetipos estratégicos por combinación de componentes"))

archetype_counts = archetype_display["archetype"].value_counts().reset_index()
archetype_counts.columns = ["archetype", "n_countries"]
display(style_table(archetype_counts, gradient_cols=["n_countries"], gradient_cmap="RdYlGn_r", format_dict={"n_countries": "{:.0f}"}))


,country_iso3,archetype,topsis_score,ipc,trend,radar_score_calc,rank_radar
9,PAN,Cercano pero requiere tesis selectiva,0.506,0.942,1.000,0.686,1
6,CRI,Prioridad natural Cibest,0.524,0.835,0.763,0.641,2
8,DOM,Cercano pero requiere tesis selectiva,0.512,0.864,0.629,0.629,3
3,ESP,Atractivo estructural lejano,0.617,0.596,0.621,0.611,4
2,CHL,Atractivo estructural lejano,0.620,0.668,0.037,0.576,5
1,USA,Atractivo estructural lejano,0.707,0.341,0.278,0.554,6
4,URY,Atractivo estructural lejano,0.585,0.540,0.290,0.542,7
10,MEX,Monitoreo / oportunidad condicionada,0.491,0.703,0.303,0.535,8
16,GTM,Monitoreo / oportunidad condicionada,0.429,0.735,0.544,0.532,9
13,PER,Cercano pero requiere tesis selectiva,0.467,0.777,0.069,0.520,10


,archetype,n_countries
0,Monitoreo / oportunidad condicionada,8
1,Cercano pero requiere tesis selectiva,6
2,Atractivo estructural lejano,6
3,Momentum táctico,3
4,Prioridad natural Cibest,1



## 9. Rankings por línea: cada línea expresa una tesis de atractividad distinta

Los rankings por línea no buscan ser completamente independientes. En servicios financieros, los países con mejor institucionalidad, digitalización y profundidad financiera tienden a ser atractivos para varias líneas. La pregunta relevante es si **las razones de atractividad** cambian entre líneas.


In [398]:
df_view = results["radar_by_line"].copy() #ver una línea específica.copy()

display(style_table(
    df_view[["country_iso3", "IB", "PF", "AD", "BD", "CIB", "GLOBAL", "rank_global"]].head(25),
    gradient_cols=["IB", "PF", "AD", "BD", "CIB", "GLOBAL"],
    gradient_cmap="RdYlGn",
    format_dict={
        "IB": "{:.3f}",
        "PF": "{:.3f}",
        "AD": "{:.3f}",
        "BD": "{:.3f}",
        "CIB": "{:.3f}",
        "GLOBAL": "{:.3f}",
    },
).set_caption("Score global por línea y general"))


,country_iso3,IB,PF,AD,BD,CIB,GLOBAL,rank_global
16,PAN,0.698,0.667,0.665,0.673,0.670,0.686,1
7,CRI,0.649,0.635,0.659,0.662,0.576,0.641,2
8,DOM,0.604,0.596,0.564,0.640,0.625,0.629,3
10,ESP,0.634,0.653,0.632,0.664,0.618,0.611,4
6,CHL,0.587,0.570,0.586,0.614,0.583,0.576,5
23,USA,0.549,0.515,0.611,0.592,0.542,0.554,6
22,URY,0.542,0.531,0.562,0.558,0.496,0.542,7
14,MEX,0.479,0.477,0.452,0.571,0.526,0.535,8
11,GTM,0.528,0.469,0.416,0.486,0.495,0.532,9
17,PER,0.492,0.503,0.462,0.557,0.512,0.520,10


In [399]:

# ---------------------------------------------------------------------------
# Rankings TOPSIS por línea de negocio 
# ---------------------------------------------------------------------------
line_rankings: Dict[str, pd.DataFrame] = {}

for business_line, ranking_df in results["business_line_rankings"].items():
    tmp = ensure_country_column(ranking_df)
    tmp = tmp.sort_values("rank").copy()
    line_rankings[business_line] = tmp

    display(Markdown(f"### {business_line} — Top 12 países"))
    display(style_table(
        tmp.head(12),
        gradient_cols=["score"],
        gradient_cmap="RdYlGn",
        format_dict={"score": "{:.3f}", "rank": "{:.0f}"},
    ).set_caption(f"Ranking {business_line}"))


### IB — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.699,0.075252,0.174775,0.602323,0.674216,0.806634,0.755174,1,IB
1,CAN,0.681,0.080104,0.170897,0.700119,0.591091,0.923344,0.668285,2,IB
2,ESP,0.655,0.088037,0.167193,0.569905,0.639838,0.730944,0.736835,3,IB
3,CHL,0.638,0.089310,0.157438,0.572716,0.608585,0.740357,0.666526,4,IB
4,URY,0.586,0.098134,0.138625,0.531180,0.504116,0.774739,0.628212,5,IB
5,BHS,0.576,0.098449,0.133548,0.490480,0.575348,0.608889,0.549265,6,IB
6,BRA,0.554,0.110340,0.136969,0.525375,0.648456,0.362913,0.549839,7,IB
7,CRI,0.538,0.111596,0.129819,0.499425,0.483061,0.675389,0.542714,8,IB
8,JAM,0.527,0.109461,0.122164,0.450975,0.558373,0.499410,0.435583,9,IB
9,PAN,0.526,0.114073,0.126350,0.508816,0.563133,0.447478,0.485130,10,IB


### PF — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,ESP,0.687,0.100484,0.220706,0.601785,0.432123,0.716088,0.801321,1,PF
1,CAN,0.655,0.116246,0.221072,0.580706,0.409833,0.912720,0.743135,2,PF
2,USA,0.641,0.117727,0.210228,0.368215,0.417226,0.800436,0.772037,3,PF
3,CHL,0.611,0.120114,0.188469,0.557462,0.364585,0.739730,0.702810,4,PF
4,URY,0.567,0.127731,0.167326,0.449576,0.326588,0.772727,0.647786,5,PF
5,BRA,0.556,0.132722,0.165900,0.374663,0.389738,0.375422,0.640003,6,PF
6,ARG,0.543,0.135093,0.160193,0.240726,0.356954,0.401102,0.647275,7,PF
7,CRI,0.514,0.134877,0.142681,0.603669,0.351748,0.685725,0.543971,8,PF
8,JAM,0.481,0.146978,0.136235,0.531548,0.721890,0.515285,0.414473,9,PF
9,PAN,0.473,0.148136,0.133171,0.680916,0.325333,0.467263,0.494237,10,PF


### AD — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.801,0.063527,0.255604,0.596469,0.711694,0.863337,0.735109,1,AD
1,CAN,0.743,0.089044,0.257708,0.730481,0.670593,0.957619,0.559761,2,AD
2,ESP,0.653,0.109510,0.205775,0.588407,0.566024,0.737571,0.551622,3,AD
3,CHL,0.637,0.116081,0.203764,0.593937,0.590357,0.764572,0.483999,4,AD
4,URY,0.619,0.124463,0.202061,0.549772,0.484997,0.777732,0.418258,5,AD
5,CRI,0.554,0.141923,0.175957,0.513842,0.314317,0.698506,0.352634,6,AD
6,BHS,0.514,0.151616,0.160482,0.581412,0.579983,0.559446,0.432479,7,AD
7,BLZ,0.475,0.177780,0.160905,0.458552,0.337389,0.360680,0.602588,8,AD
8,PAN,0.470,0.162464,0.144356,0.542400,0.444668,0.428784,0.532170,9,AD
9,ARG,0.409,0.179442,0.124085,0.339557,0.387902,0.402441,0.421955,10,AD


### BD — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,USA,0.770,0.056859,0.190484,0.812200,0.727557,0.797886,0.756028,1,BD
1,ESP,0.705,0.072268,0.173080,0.664895,0.618262,0.715501,0.784127,2,BD
2,CAN,0.698,0.074505,0.172484,0.713779,0.647569,0.911762,0.688799,3,BD
3,CHL,0.684,0.075841,0.163847,0.601183,0.636636,0.739814,0.759375,4,BD
4,BRA,0.627,0.089676,0.150480,0.714709,0.668769,0.375171,0.582621,5,BD
5,URY,0.612,0.094295,0.148468,0.502265,0.528928,0.774467,0.724425,6,BD
6,ARG,0.598,0.096362,0.143240,0.581802,0.482396,0.401832,0.705888,7,BD
7,CRI,0.558,0.102914,0.130115,0.474197,0.491003,0.686019,0.636604,8,BD
8,MEX,0.550,0.107383,0.131472,0.710308,0.445688,0.335169,0.510803,9,BD
9,DOM,0.530,0.115560,0.130140,0.514239,0.503198,0.515998,0.550751,10,BD


### CIB — Top 12 países

,country_iso3,score,d_pos,d_neg,score_macro,score_financial,score_institutional,score_digital_tech,rank,business_line
0,CAN,0.720,0.072022,0.185492,0.658005,0.697329,0.940030,0.512759,1,CIB
1,USA,0.687,0.088822,0.195256,0.575674,0.749903,0.827973,0.683950,2,CIB
2,CHL,0.632,0.092199,0.158247,0.540104,0.648643,0.746662,0.459598,3,CIB
3,ESP,0.629,0.092602,0.157235,0.616682,0.600490,0.743458,0.513287,4,CIB
4,BHS,0.534,0.115963,0.133110,0.407939,0.603988,0.596803,0.412426,5,CIB
5,BRA,0.510,0.123625,0.128907,0.524212,0.557841,0.355436,0.395872,6,CIB
6,URY,0.509,0.122109,0.126572,0.394837,0.480811,0.769076,0.409129,7,CIB
7,JAM,0.505,0.122289,0.124777,0.375096,0.603285,0.492826,0.281199,8,CIB
8,DOM,0.505,0.124123,0.126544,0.446482,0.545937,0.482710,0.283899,9,CIB
9,TTO,0.486,0.127950,0.121155,0.378126,0.588911,0.410912,0.260698,10,CIB



## 10. Correlaciones entre rankings: atractivo común vs diferenciación de tesis

Una correlación alta entre líneas no es automáticamente un error. Puede significar dos cosas muy distintas:

- **Problema de configuración:** las líneas usan pesos demasiado parecidos.
- **Factor país común:** las líneas usan pesos distintos, pero los mismos países son buenos en varias dimensiones.

Por eso comparamos **correlación de rankings** y **similitud coseno entre vectores de pesos**.


In [400]:

# ---------------------------------------------------------------------------
# Auditoría de pesos por línea
# ---------------------------------------------------------------------------
def audit_business_line_weights(
    configs: Dict[str, Dict[str, Any]],
    decision_matrix: pd.DataFrame,
) -> pd.DataFrame:
    """Audita pesos efectivos usados por TOPSIS para cada línea de negocio."""
    business_lines = configs["business_lines"]["business_lines"]
    global_dim_weights = configs["weights"]["dimension_weights"]
    global_variable_weights = configs["weights"]["variable_weights"]

    rows: List[Dict[str, Any]] = []

    for bl_key, bl_cfg in business_lines.items():
        dim_weights_line, final_var_weights = _build_business_line_weights(
            business_line_cfg=bl_cfg,
            variable_weights_by_dim=global_variable_weights,
        )
        final_var_weights_filtered = {
            var: weight
            for var, weight in final_var_weights.items()
            if var in decision_matrix.columns
        }
        total_filtered = sum(final_var_weights_filtered.values())
        if total_filtered > 0:
            final_var_weights_filtered = {
                var: weight / total_filtered
                for var, weight in final_var_weights_filtered.items()
            }
        overrides = bl_cfg.get("variable_weight_overrides", {}) or {}

        for dim, vars_global in global_variable_weights.items():
            for var, global_var_weight in vars_global.items():
                override_weight = overrides.get(dim, {}).get(var)
                rows.append({
                    "business_line": bl_key,
                    "dimension": dim,
                    "variable": var,
                    "in_decision_matrix": var in decision_matrix.columns,
                    "global_dimension_weight": global_dim_weights.get(dim),
                    "line_dimension_weight": dim_weights_line.get(dim, 0.0),
                    "global_variable_weight_in_dim": global_var_weight,
                    "override_variable_weight_in_dim": override_weight,
                    "has_override": override_weight is not None,
                    "final_topsis_weight": final_var_weights_filtered.get(var, 0.0),
                })

    return pd.DataFrame(rows)

weights_audit = audit_business_line_weights(configs, decision_matrix)

weight_sum_check = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .groupby("business_line")["final_topsis_weight"]
    .sum()
    .reset_index()
)

display(style_table(weight_sum_check, format_dict={"final_topsis_weight": "{:.6f}"}).set_caption("Validación: pesos finales por línea suman 1"))

missing_weighted = weights_audit[(~weights_audit["in_decision_matrix"]) & (weights_audit["final_topsis_weight"] > 0)]
if not missing_weighted.empty:
    risk_box("Variables ponderadas ausentes", "Hay variables ponderadas que no están en decision_matrix. Revisar configuración o extracción.")
    display(missing_weighted)
else:
    success_box("Auditoría de pesos", "No hay variables ponderadas ausentes en la matriz de decisión.")


,business_line,final_topsis_weight
0,AD,1.000000
1,BD,1.000000
2,CIB,1.000000
3,IB,1.000000
4,PF,1.000000


In [401]:

# ---------------------------------------------------------------------------
# Correlaciones de rankings y similitud de pesos
# ---------------------------------------------------------------------------
rank_vectors = {}
for line, df in line_rankings.items():
    rank_vectors[line] = df.set_index("country_iso3")["rank"]
rank_df = pd.DataFrame(rank_vectors)
rank_corr = rank_df.corr(method="spearman")

display(style_table(
    rank_corr,
    gradient_cols=rank_corr.columns.tolist(),
    gradient_cmap="YlOrRd",
    format_dict={col: "{:.3f}" for col in rank_corr.columns},
).set_caption("Correlación Spearman entre rankings por línea"))

weight_vectors = (
    weights_audit[weights_audit["in_decision_matrix"]]
    .pivot_table(
        index="business_line",
        columns="variable",
        values="final_topsis_weight",
        aggfunc="sum",
        fill_value=0.0,
    )
)

weight_cosine = pd.DataFrame(
    cosine_similarity(weight_vectors),
    index=weight_vectors.index,
    columns=weight_vectors.index,
)

display(style_table(
    weight_cosine,
    gradient_cols=weight_cosine.columns.tolist(),
    gradient_cmap="YlOrRd",
    format_dict={col: "{:.3f}" for col in weight_cosine.columns},
).set_caption("Similitud coseno entre vectores de pesos por línea"))


,IB,PF,AD,BD,CIB
IB,1.000,0.748,0.770,0.677,0.852
PF,0.748,1.000,0.865,0.890,0.687
AD,0.770,0.865,1.000,0.823,0.737
BD,0.677,0.890,0.823,1.000,0.760
CIB,0.852,0.687,0.737,0.760,1.000


business_line,AD,BD,CIB,IB,PF
business_line,,,,,
AD,1.000,0.516,0.508,0.578,0.372
BD,0.516,1.000,0.469,0.621,0.609
CIB,0.508,0.469,1.000,0.755,0.292
IB,0.578,0.621,0.755,1.000,0.367
PF,0.372,0.609,0.292,0.367,1.000


**Lectura ejecutiva.** Una correlación alta no implica error. Puede indicar que dos líneas comparten fundamentos país. La meta no es forzar rankings independientes, sino que cada línea responda a una tesis de negocio defendible.

---

Interpretación:
- Alta similitud de pesos + alta correlación de ranking:
    el problema está en configuración.

- Baja similitud de pesos + alta correlación de ranking:
    los países tienen estructura subyacente común.

In [402]:

# ---------------------------------------------------------------------------
# Lectura automática de correlación vs similitud de pesos
# ---------------------------------------------------------------------------
def pairwise_interpretation(rank_corr: pd.DataFrame, weight_cosine: pd.DataFrame) -> pd.DataFrame:
    """Cruza correlación de rankings y similitud de pesos para interpretar pares de líneas."""
    rows: List[Dict[str, Any]] = []
    lines = sorted(rank_corr.columns)

    for i, line_a in enumerate(lines):
        for line_b in lines[i + 1:]:
            corr = rank_corr.loc[line_a, line_b]
            cosine = weight_cosine.loc[line_a, line_b]
            if corr >= 0.85 and cosine >= 0.70:
                diagnosis = "Riesgo de redundancia de configuración"
            elif corr >= 0.85 and cosine < 0.70:
                diagnosis = "Factor país común con tesis diferenciadas"
            elif corr < 0.70 and cosine < 0.50:
                diagnosis = "Tesis claramente diferenciadas"
            else:
                diagnosis = "Relación moderada / esperada"

            rows.append({
                "line_a": line_a,
                "line_b": line_b,
                "rank_corr": corr,
                "weight_cosine": cosine,
                "diagnosis": diagnosis,
            })

    return pd.DataFrame(rows).sort_values("rank_corr", ascending=False)

pair_diagnosis = pairwise_interpretation(rank_corr, weight_cosine)

display(style_table(
    pair_diagnosis,
    gradient_cols=["rank_corr", "weight_cosine"],
    gradient_cmap="YlOrRd",
    format_dict={"rank_corr": "{:.3f}", "weight_cosine": "{:.3f}"},
).set_caption("Diagnóstico: correlación de ranking vs similitud de pesos"))

common_factor_pairs = pair_diagnosis.query("diagnosis == 'Factor país común con tesis diferenciadas'")
if not common_factor_pairs.empty:
    insight_box(
        "Atractivo común entre países",
        "Algunos rankings correlacionan alto aunque sus pesos sean distintos. Esto indica que ciertos países concentran atributos transversales —institucionalidad, digitalización, profundidad financiera y estabilidad— que los hacen atractivos para varias líneas a la vez."
    )


,line_a,line_b,rank_corr,weight_cosine,diagnosis
6,BD,PF,0.890,0.609,Factor país común con tesis diferenciadas
3,AD,PF,0.865,0.372,Factor país común con tesis diferenciadas
7,CIB,IB,0.852,0.755,Riesgo de redundancia de configuración
0,AD,BD,0.823,0.516,Relación moderada / esperada
2,AD,IB,0.770,0.578,Relación moderada / esperada
4,BD,CIB,0.760,0.469,Relación moderada / esperada
9,IB,PF,0.748,0.367,Relación moderada / esperada
1,AD,CIB,0.737,0.508,Relación moderada / esperada
8,CIB,PF,0.687,0.292,Tesis claramente diferenciadas
5,BD,IB,0.677,0.621,Relación moderada / esperada



## 11. Variables que realmente diferencian las tesis

La diferenciación de tesis no se evalúa solo con pesos por dimensión. Debe observarse qué variables tienen mayor dispersión de peso entre líneas. Estas variables son los verdaderos “separadores estratégicos” del modelo.


In [403]:

# ---------------------------------------------------------------------------
# Spread de pesos entre líneas
# ---------------------------------------------------------------------------
weight_spread = weight_vectors.T.copy()
weight_spread["max_weight"] = weight_spread.max(axis=1)
weight_spread["min_weight"] = weight_spread.min(axis=1)
weight_spread["spread"] = weight_spread["max_weight"] - weight_spread["min_weight"]
weight_spread["dominant_line"] = weight_vectors.idxmax(axis=0)
weight_spread = weight_spread.sort_values("spread", ascending=False).reset_index().rename(columns={"index": "variable"})

display(style_table(
    weight_spread.head(20),
    gradient_cols=["spread", "max_weight"],
    gradient_cmap="YlOrRd",
    format_dict={col: "{:.4f}" for col in weight_vectors.index.tolist() + ["max_weight", "min_weight", "spread"] if col in weight_spread.columns},
).set_caption("Variables que más diferencian las tesis de negocio"))

fig = px.bar(
    weight_spread.head(15).sort_values("spread"),
    x="spread",
    y="variable",
    orientation="h",
    color="dominant_line",
    title="Variables con mayor dispersión de peso entre líneas",
)
fig.update_layout(xaxis_title="Spread de peso", yaxis_title="Variable")
fig.show()


business_line,variable,AD,BD,CIB,IB,PF,max_weight,min_weight,spread,dominant_line
0,digital_payment_adults_pct,0.0264,0.0540,0.0020,0.0080,0.1880,0.1880,0.0020,0.1860,PF
1,regulatory_quality,0.1632,0.0330,0.0552,0.0630,0.0276,0.1632,0.0276,0.1356,AD
2,secure_internet_servers_per_million,0.1320,0.0480,0.0060,0.0120,0.0235,0.1320,0.0060,0.1260,AD
3,stock_market_cap_gdp,0.0312,0.0025,0.1200,0.0068,0.0050,0.1200,0.0025,0.1175,CIB
4,rule_of_law,0.1248,0.0270,0.0624,0.0690,0.0216,0.1248,0.0216,0.1032,AD
5,internet_users_pct,0.0462,0.1020,0.0034,0.0110,0.0517,0.1020,0.0034,0.0986,BD
6,mobile_subscriptions,0.0231,0.0660,0.0016,0.0090,0.0846,0.0846,0.0016,0.0830,PF
7,personal_remittances_gdp,0.0036,0.0050,0.0120,0.0023,0.0850,0.0850,0.0023,0.0828,PF
8,population_total,0.0014,0.0840,0.0136,0.0140,0.0048,0.0840,0.0014,0.0826,BD
9,ict_goods_exports_pct_total_goods_exports,0.0825,0.0150,0.0056,0.0025,0.0188,0.0825,0.0025,0.0800,AD



## 12. Top-N overlap: similitud decisional, no solo estadística

La correlación resume todo el ranking, pero la decisión ejecutiva suele concentrarse en Top 5 o Top 10. Dos líneas pueden correlacionar alto y aun así recomendar prioridades diferentes en el tramo relevante.


In [404]:

# ---------------------------------------------------------------------------
# Overlap Top-N entre líneas
# ---------------------------------------------------------------------------
def topn_overlap(rankings_by_line: Dict[str, pd.DataFrame], n: int = 5) -> pd.DataFrame:
    """Calcula solapamiento de Top-N entre rankings de líneas."""
    rows: List[Dict[str, Any]] = []
    lines = list(rankings_by_line.keys())

    for i, line_a in enumerate(lines):
        for line_b in lines[i + 1:]:
            a = rankings_by_line[line_a].sort_values("rank").head(n)["country_iso3"].tolist()
            b = rankings_by_line[line_b].sort_values("rank").head(n)["country_iso3"].tolist()
            overlap = len(set(a) & set(b))
            rows.append({
                "line_a": line_a,
                "line_b": line_b,
                "n": n,
                "overlap": overlap,
                "overlap_pct": overlap / n,
                "top_a": ", ".join(a),
                "top_b": ", ".join(b),
            })

    return pd.DataFrame(rows).sort_values("overlap_pct", ascending=False)

top5_overlap = topn_overlap(line_rankings, n=5)
top10_overlap = topn_overlap(line_rankings, n=10)

display(style_table(
    top5_overlap,
    gradient_cols=["overlap_pct"],
    gradient_cmap="YlOrRd",
    format_dict={"overlap_pct": "{:.0%}"},
).set_caption("Solapamiento Top 5 entre líneas"))

display(style_table(
    top10_overlap,
    gradient_cols=["overlap_pct"],
    gradient_cmap="YlOrRd",
    format_dict={"overlap_pct": "{:.0%}"},
).set_caption("Solapamiento Top 10 entre líneas"))


,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
0,IB,PF,5,5,100%,"USA, CAN, ESP, CHL, URY","ESP, CAN, USA, CHL, URY"
1,IB,AD,5,5,100%,"USA, CAN, ESP, CHL, URY","USA, CAN, ESP, CHL, URY"
4,PF,AD,5,5,100%,"ESP, CAN, USA, CHL, URY","USA, CAN, ESP, CHL, URY"
2,IB,BD,5,4,80%,"USA, CAN, ESP, CHL, URY","USA, ESP, CAN, CHL, BRA"
3,IB,CIB,5,4,80%,"USA, CAN, ESP, CHL, URY","CAN, USA, CHL, ESP, BHS"
5,PF,BD,5,4,80%,"ESP, CAN, USA, CHL, URY","USA, ESP, CAN, CHL, BRA"
6,PF,CIB,5,4,80%,"ESP, CAN, USA, CHL, URY","CAN, USA, CHL, ESP, BHS"
7,AD,BD,5,4,80%,"USA, CAN, ESP, CHL, URY","USA, ESP, CAN, CHL, BRA"
8,AD,CIB,5,4,80%,"USA, CAN, ESP, CHL, URY","CAN, USA, CHL, ESP, BHS"
9,BD,CIB,5,4,80%,"USA, ESP, CAN, CHL, BRA","CAN, USA, CHL, ESP, BHS"


,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
0,IB,PF,10,9,90%,"USA, CAN, ESP, CHL, URY, BHS, BRA, CRI, JAM, PAN","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, PAN"
1,IB,AD,10,8,80%,"USA, CAN, ESP, CHL, URY, BHS, BRA, CRI, JAM, PAN","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
3,IB,CIB,10,8,80%,"USA, CAN, ESP, CHL, URY, BHS, BRA, CRI, JAM, PAN","CAN, USA, CHL, ESP, BHS, BRA, URY, JAM, DOM, TTO"
4,PF,AD,10,8,80%,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, PAN","USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG"
5,PF,BD,10,8,80%,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, PAN","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, DOM"
2,IB,BD,10,7,70%,"USA, CAN, ESP, CHL, URY, BHS, BRA, CRI, JAM, PAN","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, DOM"
6,PF,CIB,10,7,70%,"ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, PAN","CAN, USA, CHL, ESP, BHS, BRA, URY, JAM, DOM, TTO"
7,AD,BD,10,7,70%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, DOM"
9,BD,CIB,10,7,70%,"USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, DOM","CAN, USA, CHL, ESP, BHS, BRA, URY, JAM, DOM, TTO"
8,AD,CIB,10,6,60%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","CAN, USA, CHL, ESP, BHS, BRA, URY, JAM, DOM, TTO"



## 13. Bandas competitivas y gaps: evitar decisiones por diferencias marginales

No toda diferencia ordinal es material. Si dos países tienen scores casi iguales, el ranking debe comunicarse como una banda competitiva. Esto evita sobreinterpretar diferencias que pueden cambiar con pequeñas variaciones de pesos, imputación o universo de países.


In [405]:

# ---------------------------------------------------------------------------
# Bandas y gaps por línea
# ---------------------------------------------------------------------------
def classify_gap(gap: float) -> str:
    """Clasifica la materialidad de un gap de score."""
    if pd.isna(gap):
        return "Sin siguiente país"
    if gap < 0.005:
        return "Empate práctico"
    if gap < 0.015:
        return "Diferencia débil"
    if gap < 0.030:
        return "Diferencia moderada"
    return "Diferencia material"


def assign_tier_by_score(score: float, q80: float, q60: float, q40: float) -> str:
    """Asigna banda de atractividad relativa por cuantiles."""
    if score >= q80:
        return "Alta"
    if score >= q60:
        return "Media-alta"
    if score >= q40:
        return "Media"
    return "Baja"


def strategic_read(tier: str, gap: str) -> str:
    """Genera lectura estratégica combinando banda y gap."""
    if tier == "Alta" and gap == "Diferencia material":
        return "Liderazgo claramente diferenciado"
    if tier == "Alta" and gap in {"Empate práctico", "Diferencia débil"}:
        return "Alta atractividad, pero no distinguible del siguiente"
    if tier == "Media-alta":
        return "Banda competitiva media-alta; requiere tesis específica"
    if tier == "Media":
        return "Atractividad intermedia; priorizar solo si hay racional estratégico"
    return "Baja prioridad relativa en el universo evaluado"

line_tier_tables: Dict[str, pd.DataFrame] = {}

for line, df in line_rankings.items():
    tmp = df.sort_values("score", ascending=False).copy()
    q80, q60, q40 = tmp["score"].quantile([0.80, 0.60, 0.40]).tolist()
    tmp["attractiveness_tier"] = tmp["score"].apply(lambda x: assign_tier_by_score(x, q80, q60, q40))
    tmp["score_gap_next"] = tmp["score"] - tmp["score"].shift(-1)
    tmp["gap_interpretation"] = tmp["score_gap_next"].apply(classify_gap)
    tmp["strategic_read"] = tmp.apply(lambda r: strategic_read(r["attractiveness_tier"], r["gap_interpretation"]), axis=1)
    line_tier_tables[line] = tmp

for line, tmp in line_tier_tables.items():
    display(Markdown(f"### {line} — Bandas y gaps"))
    display(style_table(
        tmp[["country_iso3", "score", "rank", "attractiveness_tier", "score_gap_next", "gap_interpretation", "strategic_read"]].head(15),
        gradient_cols=["score", "score_gap_next"],
        gradient_cmap="RdYlGn",
        format_dict={"score": "{:.3f}", "rank": "{:.0f}", "score_gap_next": "{:.3f}"},
    ))


### IB — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.699,1,Alta,0.018,Diferencia moderada,Baja prioridad relativa en el universo evaluado
1,CAN,0.681,2,Alta,0.026,Diferencia moderada,Baja prioridad relativa en el universo evaluado
2,ESP,0.655,3,Alta,0.017,Diferencia moderada,Baja prioridad relativa en el universo evaluado
3,CHL,0.638,4,Alta,0.053,Diferencia material,Liderazgo claramente diferenciado
4,URY,0.586,5,Alta,0.010,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
5,BHS,0.576,6,Media-alta,0.022,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
6,BRA,0.554,7,Media-alta,0.016,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
7,CRI,0.538,8,Media-alta,0.010,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
8,JAM,0.527,9,Media-alta,0.002,Empate práctico,Banda competitiva media-alta; requiere tesis específica
9,PAN,0.526,10,Media-alta,0.021,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica


### PF — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,ESP,0.687,1,Alta,0.032,Diferencia material,Liderazgo claramente diferenciado
1,CAN,0.655,2,Alta,0.014,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
2,USA,0.641,3,Alta,0.030,Diferencia material,Liderazgo claramente diferenciado
3,CHL,0.611,4,Alta,0.044,Diferencia material,Liderazgo claramente diferenciado
4,URY,0.567,5,Alta,0.012,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
5,BRA,0.556,6,Media-alta,0.013,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
6,ARG,0.543,7,Media-alta,0.028,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
7,CRI,0.514,8,Media-alta,0.033,Diferencia material,Banda competitiva media-alta; requiere tesis específica
8,JAM,0.481,9,Media-alta,0.008,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
9,PAN,0.473,10,Media-alta,0.005,Empate práctico,Banda competitiva media-alta; requiere tesis específica


### AD — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.801,1,Alta,0.058,Diferencia material,Liderazgo claramente diferenciado
1,CAN,0.743,2,Alta,0.091,Diferencia material,Liderazgo claramente diferenciado
2,ESP,0.653,3,Alta,0.016,Diferencia moderada,Baja prioridad relativa en el universo evaluado
3,CHL,0.637,4,Alta,0.018,Diferencia moderada,Baja prioridad relativa en el universo evaluado
4,URY,0.619,5,Alta,0.065,Diferencia material,Liderazgo claramente diferenciado
5,CRI,0.554,6,Media-alta,0.039,Diferencia material,Banda competitiva media-alta; requiere tesis específica
6,BHS,0.514,7,Media-alta,0.039,Diferencia material,Banda competitiva media-alta; requiere tesis específica
7,BLZ,0.475,8,Media-alta,0.005,Empate práctico,Banda competitiva media-alta; requiere tesis específica
8,PAN,0.470,9,Media-alta,0.062,Diferencia material,Banda competitiva media-alta; requiere tesis específica
9,ARG,0.409,10,Media-alta,0.005,Empate práctico,Banda competitiva media-alta; requiere tesis específica


### BD — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,USA,0.770,1,Alta,0.065,Diferencia material,Liderazgo claramente diferenciado
1,ESP,0.705,2,Alta,0.007,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
2,CAN,0.698,3,Alta,0.015,Diferencia débil,"Alta atractividad, pero no distinguible del siguiente"
3,CHL,0.684,4,Alta,0.057,Diferencia material,Liderazgo claramente diferenciado
4,BRA,0.627,5,Alta,0.015,Diferencia moderada,Baja prioridad relativa en el universo evaluado
5,URY,0.612,6,Media-alta,0.014,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
6,ARG,0.598,7,Media-alta,0.039,Diferencia material,Banda competitiva media-alta; requiere tesis específica
7,CRI,0.558,8,Media-alta,0.008,Diferencia débil,Banda competitiva media-alta; requiere tesis específica
8,MEX,0.550,9,Media-alta,0.021,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
9,DOM,0.530,10,Media-alta,0.002,Empate práctico,Banda competitiva media-alta; requiere tesis específica


### CIB — Bandas y gaps

,country_iso3,score,rank,attractiveness_tier,score_gap_next,gap_interpretation,strategic_read
0,CAN,0.720,1,Alta,0.033,Diferencia material,Liderazgo claramente diferenciado
1,USA,0.687,2,Alta,0.055,Diferencia material,Liderazgo claramente diferenciado
2,CHL,0.632,3,Alta,0.003,Empate práctico,"Alta atractividad, pero no distinguible del siguiente"
3,ESP,0.629,4,Alta,0.095,Diferencia material,Liderazgo claramente diferenciado
4,BHS,0.534,5,Alta,0.024,Diferencia moderada,Baja prioridad relativa en el universo evaluado
5,BRA,0.510,6,Media-alta,0.001,Empate práctico,Banda competitiva media-alta; requiere tesis específica
6,URY,0.509,7,Media-alta,0.004,Empate práctico,Banda competitiva media-alta; requiere tesis específica
7,JAM,0.505,8,Media-alta,0.000,Empate práctico,Banda competitiva media-alta; requiere tesis específica
8,DOM,0.505,9,Media-alta,0.018,Diferencia moderada,Banda competitiva media-alta; requiere tesis específica
9,TTO,0.486,10,Media-alta,0.008,Diferencia débil,Banda competitiva media-alta; requiere tesis específica



## 14. Dispersión de países entre líneas: mercados multi-tesis vs apuestas selectivas

Un país con posiciones similares en varias líneas es un mercado transversalmente atractivo. Un país con alta dispersión entre líneas no es “malo”; es un mercado cuya atractividad depende de una tesis específica.


In [406]:

# ---------------------------------------------------------------------------
# Dispersión de ranking por país entre líneas
# ---------------------------------------------------------------------------
rank_by_country = pd.DataFrame({
    line: df.set_index("country_iso3")["rank"]
    for line, df in line_rankings.items()
})

rank_dispersion = rank_by_country.copy()
rank_dispersion["rank_mean_across_lines"] = rank_dispersion.mean(axis=1)
rank_dispersion["rank_std_across_lines"] = rank_dispersion.std(axis=1)
rank_dispersion["rank_range_across_lines"] = rank_dispersion.max(axis=1) - rank_dispersion.min(axis=1)
rank_dispersion = rank_dispersion.sort_values("rank_range_across_lines", ascending=False).reset_index().rename(columns={"index": "country_iso3"})

display(style_table(
    rank_dispersion.head(20),
    gradient_cols=["rank_std_across_lines", "rank_range_across_lines"],
    gradient_cmap="YlOrRd",
    format_dict={"rank_mean_across_lines": "{:.1f}", "rank_std_across_lines": "{:.1f}", "rank_range_across_lines": "{:.0f}"},
).set_caption("Países con mayor sensibilidad a la tesis de negocio"))

high_dispersion = rank_dispersion.head(5)["country_iso3"].tolist()
low_dispersion = rank_dispersion.sort_values("rank_range_across_lines").head(5)["country_iso3"].tolist()

display(Markdown(f"""
### Lectura de dispersión

- **Países más dependientes de tesis específica:** {', '.join(high_dispersion)}.  
  Requieren conversación por línea: pueden ser muy atractivos para una tesis y poco atractivos para otra.

- **Países con atractividad transversal más estable:** {', '.join(low_dispersion)}.  
  Son candidatos a discusiones de portafolio regional o multi-línea.
"""))


,country_iso3,IB,PF,AD,BD,CIB,rank_mean_across_lines,rank_std_across_lines,rank_range_across_lines
0,NIC,24,24,24,24,24,24.0,0.0,24
1,GTM,18,23,20,22,22,21.0,1.8,21
2,HND,16,22,22,23,19,20.4,2.6,20
3,BOL,15,21,23,18,17,18.8,2.9,20
4,ECU,20,20,21,19,15,19.0,2.1,19
5,SUR,22,11,18,16,21,17.6,3.9,18
6,MEX,23,19,15,9,12,15.6,5.0,18
7,SLV,14,17,19,20,20,18.0,2.3,18
8,BLZ,17,16,8,21,18,16.0,4.3,17
9,PER,19,15,14,11,13,14.4,2.7,16



### Lectura de dispersión

- **Países más dependientes de tesis específica:** NIC, GTM, HND, BOL, ECU.  
  Requieren conversación por línea: pueden ser muy atractivos para una tesis y poco atractivos para otra.

- **Países con atractividad transversal más estable:** USA, CAN, ESP, CHL, URY.  
  Son candidatos a discusiones de portafolio regional o multi-línea.



## 15. Explicabilidad: contribución ponderada + efecto marginal

La contribución ponderada ayuda a explicar el perfil de un país, pero TOPSIS no es lineal. Por eso complementamos con análisis marginal leave-one-variable-out. La combinación permite distinguir:

- **Driver robusto:** alta contribución y efecto marginal positivo.
- **Driver descriptivo:** alta contribución pero bajo efecto marginal.
- **Restricción crítica:** alto shortfall y efecto marginal negativo.
- **Efecto marginal bajo:** variable poco decisiva para el ranking.


In [407]:

# ---------------------------------------------------------------------------
# Explicabilidad: contribuciones y marginales
# ---------------------------------------------------------------------------
contrib_by_line = compute_all_business_line_contributions(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
)

marginal_by_line = compute_all_marginal_effects(
    decision_matrix=decision_matrix,
    weights_audit=weights_audit,
    variable_catalog=variable_catalog,
    distance_metric=configs["settings"]["scoring"]["topsis"].get("distance_metric", "euclidean"),
)

explainability_by_line: Dict[str, pd.DataFrame] = {}
for line in line_rankings.keys():
    tmp = combine_contribution_and_marginal(contrib_by_line[line], marginal_by_line[line])
    tmp["driver_class"] = tmp.apply(classify_driver_robustness, axis=1)
    explainability_by_line[line] = tmp

success_box("Explicabilidad calculada", "Se combinaron contribuciones ponderadas y efectos marginales por línea de negocio.")


2026-06-26 00:55:27 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.801 (USA)
--- Logging error in Loguru Handler #26 ---
Record was: {'elapsed': datetime.timedelta(days=2, seconds=42009, microseconds=393633), 'exception': None, 'extra': {'n': 25, 's': 0.8010396830192501, 'c': 'USA'}, 'file': (name='ranking.py', path='c:\\Users\\jadarve\\OneDrive - Grupo Bancolombia\\Bancolombia\\GFEC-VEF\\2026\\Internacionalización\\radar_cibest_v2\\src\\scoring\\ranking.py'), 'function': 'rank', 'level': (name='INFO', no=20, icon='ℹ️'), 'line': 133, 'message': 'TOPSIS completado: 25 paises | score max=0.801 (USA)', 'module': 'ranking', 'name': 'src.scoring.ranking', 'process': (id=24300, name='MainProcess'), 'thread': (id=37796, name='MainThread'), 'time': datetime(2026, 6, 26, 0, 55, 27, 69595, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Hora est. Pacífico, Sudamérica'))}
Traceback (most recent call last):
  File "C:\Users\jadarve\AppD

In [408]:

# ---------------------------------------------------------------------------
# Drivers por línea para el país líder y países prioritarios
# ---------------------------------------------------------------------------
def show_country_drivers(country_iso3: str, line: str, top_n: int = 8) -> None:
    """Muestra drivers y restricciones de un país en una línea específica."""
    display(Markdown(f"### Drivers y restricciones — {country_iso3} en {line}"))
    table = build_country_driver_table(
        explainability_df=explainability_by_line[line],
        country_iso3=country_iso3,
        top_n=top_n,
    )
    display(style_table(
        table,
        gradient_cols=["contribution", "shortfall", "score_effect"],
        gradient_cmap="RdYlGn",
        format_dict={
            "normalized_value": "{:.3f}",
            "contribution": "{:.4f}",
            "shortfall": "{:.4f}",
            "score_effect": "{:.4f}",
            "rank_effect": "{:+.0f}",
        },
    ))

# Mostrar drivers del líder de cada línea.
for line, df in line_rankings.items():
    leader_country = df.sort_values("rank").iloc[0]["country_iso3"]
    show_country_drivers(leader_country, line, top_n=8)


### Drivers y restricciones — USA en IB

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,IB,USA,financial_system_deposits_gdp,financial,0.880,0.081000,0.0712,0.0098,0.0175,+1,Driver,Driver,Driver robusto
1,IB,USA,regulatory_quality,institutional,0.996,0.063000,0.0628,0.0002,0.0147,+0,Driver,Driver,Driver robusto
2,IB,USA,rule_of_law,institutional,0.838,0.069000,0.0578,0.0112,0.0099,+0,Driver,Driver,Driver descriptivo
3,IB,USA,bank_npl_ratio,financial,0.912,0.058500,0.0534,0.0051,0.0099,+0,Driver,Driver,Driver descriptivo
4,IB,USA,account_ownership,financial,0.982,0.045000,0.0442,0.0008,0.0070,+0,Driver,Driver,Driver descriptivo
5,IB,USA,government_effectiveness,institutional,0.858,0.051000,0.0438,0.0072,0.0058,+0,Driver,Driver,Driver descriptivo
6,IB,USA,control_of_corruption,institutional,0.822,0.045000,0.0370,0.0080,0.0036,+0,Driver,Driver,Driver descriptivo
7,IB,USA,gdp_per_capita_ppp,macro,1.000,0.032000,0.0320,0.0000,0.0036,+0,Driver,Driver,Driver descriptivo
8,IB,USA,bank_capital_rwa,financial,0.189,0.054000,0.0102,0.0438,-0.0413,+0,Restriccion,Restriccion,Restriccion critica
9,IB,USA,public_debt_gdp,macro,0.000,0.028000,0.0000,0.0280,-0.0154,+0,Restriccion,Restriccion,Restriccion critica


### Drivers y restricciones — ESP en PF

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,PF,ESP,digital_payment_adults_pct,digital_tech,0.990,0.188000,0.1861,0.0019,0.1456,+2,Driver,Driver,Driver robusto
1,PF,ESP,account_ownership,financial,1.000,0.055000,0.0550,0.0000,0.0069,+0,Driver,Driver,Driver descriptivo
2,PF,ESP,internet_users_pct,digital_tech,1.000,0.051700,0.0517,0.0000,0.0061,+0,Driver,Driver,Driver descriptivo
3,PF,ESP,inflation_rate,macro,0.986,0.022400,0.0221,0.0003,0.0011,+0,Driver,Driver,Efecto marginal bajo
4,PF,ESP,financial_system_deposits_gdp,financial,1.000,0.020000,0.0200,0.0000,0.0009,+0,Driver,Driver,Efecto marginal bajo
5,PF,ESP,commercial_bank_branches_per_100k_adults,digital_tech,0.732,0.042300,0.0310,0.0113,0.0008,+0,Driver,Driver,Driver descriptivo
6,PF,ESP,rule_of_law,institutional,0.828,0.021600,0.0179,0.0037,0.0006,+0,Driver,Driver,Efecto marginal bajo
7,PF,ESP,weighted_mean_applied_tariff_all_products,macro,0.951,0.016000,0.0152,0.0008,0.0005,+0,Driver,Driver,Efecto marginal bajo
8,PF,ESP,personal_remittances_gdp,financial,0.086,0.085000,0.0073,0.0777,-0.0887,+0,Restriccion,Restriccion,Restriccion critica
9,PF,ESP,mobile_subscriptions,digital_tech,0.577,0.084600,0.0488,0.0358,-0.0091,+0,Restriccion,Restriccion,Driver descriptivo


### Drivers y restricciones — USA en AD

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,AD,USA,regulatory_quality,institutional,0.996,0.163200,0.1626,0.0006,0.0445,+0,Driver,Driver,Driver robusto
1,AD,USA,secure_internet_servers_per_million,digital_tech,0.921,0.132000,0.1215,0.0105,0.0188,+0,Driver,Driver,Driver robusto
2,AD,USA,rule_of_law,institutional,0.838,0.124800,0.1046,0.0202,0.0061,+0,Driver,Driver,Driver descriptivo
3,AD,USA,internet_users_pct,digital_tech,0.971,0.046200,0.0449,0.0013,0.0025,+0,Driver,Driver,Driver descriptivo
4,AD,USA,control_of_corruption,institutional,0.822,0.086400,0.0710,0.0154,0.0016,+0,Driver,Driver,Driver descriptivo
5,AD,USA,stock_market_cap_gdp,financial,1.000,0.031200,0.0312,0.0000,0.0012,+0,Driver,Driver,Driver descriptivo
6,AD,USA,government_effectiveness,institutional,0.858,0.048000,0.0412,0.0068,0.0012,+0,Driver,Driver,Driver descriptivo
7,AD,USA,digital_payment_adults_pct,digital_tech,0.934,0.026400,0.0247,0.0017,0.0007,+0,Driver,Driver,Efecto marginal bajo
8,AD,USA,ict_goods_exports_pct_total_goods_exports,digital_tech,0.440,0.082500,0.0363,0.0462,-0.0522,+0,Restriccion,Restriccion,Restriccion critica
9,AD,USA,political_stability,institutional,0.389,0.033600,0.0131,0.0205,-0.0084,+0,Restriccion,Restriccion,Driver moderado


### Drivers y restricciones — USA en BD

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,BD,USA,internet_users_pct,digital_tech,0.971,0.102000,0.0991,0.0029,0.0288,+0,Driver,Driver,Driver robusto
1,BD,USA,population_total,macro,1.000,0.084000,0.0840,0.0000,0.0197,+0,Driver,Driver,Driver robusto
2,BD,USA,digital_payment_adults_pct,digital_tech,0.934,0.054000,0.0505,0.0035,0.0061,+0,Driver,Driver,Driver descriptivo
3,BD,USA,account_ownership,financial,0.982,0.050000,0.0491,0.0009,0.0061,+0,Driver,Driver,Driver descriptivo
4,BD,USA,secure_internet_servers_per_million,digital_tech,0.921,0.048000,0.0442,0.0038,0.0045,+0,Driver,Driver,Driver descriptivo
5,BD,USA,gdp_per_capita_ppp,macro,1.000,0.042000,0.0420,0.0000,0.0044,+0,Driver,Driver,Driver descriptivo
6,BD,USA,gdp_nominal,macro,1.000,0.036000,0.0360,0.0000,0.0032,+0,Driver,Driver,Driver descriptivo
7,BD,USA,inflation_rate,macro,0.985,0.036000,0.0355,0.0005,0.0031,+0,Driver,Driver,Driver descriptivo
8,BD,USA,mobile_subscriptions,digital_tech,0.421,0.066000,0.0278,0.0382,-0.0473,+0,Restriccion,Restriccion,Restriccion critica
9,BD,USA,public_debt_gdp,macro,0.000,0.021000,0.0000,0.0210,-0.0127,+0,Restriccion,Restriccion,Restriccion critica


### Drivers y restricciones — CAN en CIB

,business_line,country_iso3,removed_variable,dimension_contribution,normalized_value,final_topsis_weight_contribution,contribution,shortfall,score_effect,rank_effect,effect_type,driver_side,driver_class
0,CIB,CAN,stock_market_cap_gdp,financial,0.925,0.120000,0.1110,0.0090,0.0449,+0,Driver,Driver,Driver robusto
1,CIB,CAN,rule_of_law,institutional,1.000,0.062400,0.0624,0.0000,0.0122,+0,Driver,Driver,Driver robusto
2,CIB,CAN,regulatory_quality,institutional,1.000,0.055200,0.0552,0.0000,0.0094,+0,Driver,Driver,Driver descriptivo
3,CIB,CAN,government_effectiveness,institutional,1.000,0.043200,0.0432,0.0000,0.0056,+0,Driver,Driver,Driver descriptivo
4,CIB,CAN,control_of_corruption,institutional,1.000,0.036000,0.0360,0.0000,0.0039,+0,Driver,Driver,Driver descriptivo
5,CIB,CAN,bank_npl_ratio,financial,1.000,0.032000,0.0320,0.0000,0.0030,+0,Driver,Driver,Driver descriptivo
6,CIB,CAN,inflation_rate,macro,0.987,0.030600,0.0302,0.0004,0.0027,+0,Driver,Driver,Driver descriptivo
7,CIB,CAN,country_risk_premium,institutional,1.000,0.012000,0.0120,0.0000,0.0004,+0,Driver,Driver,Efecto marginal bajo
8,CIB,CAN,bank_capital_rwa,financial,0.169,0.044000,0.0074,0.0366,-0.0288,+0,Restriccion,Restriccion,Restriccion critica
9,CIB,CAN,trade_openness,macro,0.476,0.054400,0.0259,0.0285,-0.0149,+1,Restriccion,Restriccion,Restriccion critica



## 16. Lectura de tesis por línea

Esta sección traduce los rankings a lógica de negocio. El objetivo es que el Comité no vea cinco listas, sino cinco formas de contestar la pregunta: **¿qué significa que un país sea atractivo para esta línea?**


In [409]:

# ---------------------------------------------------------------------------
# Lectura ejecutiva automática por línea
# ---------------------------------------------------------------------------
BUSINESS_THESIS = {
    "IB": "Intermediación Bancaria privilegia profundidad financiera, fondeo, calidad de cartera, solvencia e institucionalidad. Un país alto en IB debe leerse como mercado bancario estructuralmente desarrollable, no necesariamente como mercado digitalmente expansivo.",
    "PF": "Pagos y Flujos privilegia comportamiento transaccional observado, pagos digitales, remesas, apertura y capilaridad operativa. Un país alto en PF indica potencial de mover dinero, recaudar, pagar y conectar flujos.",
    "AD": "Activos Digitales privilegia regulación, estado de derecho, control de corrupción, infraestructura digital segura y sofisticación financiera. Un país alto en AD debe leerse como jurisdicción más apta para innovación digital financiera bajo menor incertidumbre institucional.",
    "BD": "Bancos Digitales privilegia escala retail, población urbana, conectividad, bancarización y potencial de adquisición digital. Un país alto en BD indica potencial de escalar una base masiva de usuarios digitales.",
    "CIB": "Corporate & Investment Banking privilegia escala económica, profundidad de mercado de capitales, apertura, IED, solvencia e institucionalidad. Un país alto en CIB indica mayor potencial para banca corporativa, mercados de capitales y asset management.",
}

for line, thesis in BUSINESS_THESIS.items():
    if line not in line_rankings:
        continue
    df = line_rankings[line].sort_values("rank")
    top = df.head(5)["country_iso3"].tolist()
    display(Markdown(f"""
### {line} — Tesis de atractividad

{thesis}

**Top 5 actual:** {', '.join(top)}.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En {line}, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.
"""))



### IB — Tesis de atractividad

Intermediación Bancaria privilegia profundidad financiera, fondeo, calidad de cartera, solvencia e institucionalidad. Un país alto en IB debe leerse como mercado bancario estructuralmente desarrollable, no necesariamente como mercado digitalmente expansivo.

**Top 5 actual:** USA, CAN, ESP, CHL, URY.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En IB, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### PF — Tesis de atractividad

Pagos y Flujos privilegia comportamiento transaccional observado, pagos digitales, remesas, apertura y capilaridad operativa. Un país alto en PF indica potencial de mover dinero, recaudar, pagar y conectar flujos.

**Top 5 actual:** ESP, CAN, USA, CHL, URY.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En PF, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### AD — Tesis de atractividad

Activos Digitales privilegia regulación, estado de derecho, control de corrupción, infraestructura digital segura y sofisticación financiera. Un país alto en AD debe leerse como jurisdicción más apta para innovación digital financiera bajo menor incertidumbre institucional.

**Top 5 actual:** USA, CAN, ESP, CHL, URY.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En AD, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### BD — Tesis de atractividad

Bancos Digitales privilegia escala retail, población urbana, conectividad, bancarización y potencial de adquisición digital. Un país alto en BD indica potencial de escalar una base masiva de usuarios digitales.

**Top 5 actual:** USA, ESP, CAN, CHL, BRA.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En BD, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



### CIB — Tesis de atractividad

Corporate & Investment Banking privilegia escala económica, profundidad de mercado de capitales, apertura, IED, solvencia e institucionalidad. Un país alto en CIB indica mayor potencial para banca corporativa, mercados de capitales y asset management.

**Top 5 actual:** CAN, USA, CHL, ESP, BHS.

**Implicación de negocio:** el Top 5 no debe interpretarse igual para todas las líneas. En CIB, la priorización depende de los drivers que materializan esta tesis específica, no solo del score agregado.



## 17. Implicaciones para decisión estratégica

La salida del modelo debe traducirse en decisiones de priorización, no en una carrera por el primer puesto. Las decisiones recomendadas se estructuran en cuatro tipos.


In [410]:

# ---------------------------------------------------------------------------
# Matriz ejecutiva de implicaciones
# ---------------------------------------------------------------------------
def decision_implication(tier: str, gap: str) -> str:
    """Traduce tier y gap en recomendación ejecutiva."""
    if tier == "Alta" and gap == "Diferencia material":
        return "Priorizar análisis de entrada / business case"
    if tier == "Alta":
        return "Mantener en shortlist; diferenciar por drivers y restricciones"
    if tier == "Media-alta":
        return "Profundizar solo si existe encaje con línea específica"
    if tier == "Media":
        return "Monitorear; no priorizar sin tesis adicional"
    return "Descartar del ciclo actual salvo razón estratégica externa"

executive_rows: List[Dict[str, Any]] = []
for line, table in line_tier_tables.items():
    for _, row in table.head(10).iterrows():
        executive_rows.append({
            "business_line": line,
            "country_iso3": row["country_iso3"],
            "rank": row["rank"],
            "score": row["score"],
            "tier": row["attractiveness_tier"],
            "gap": row["gap_interpretation"],
            "decision_implication": decision_implication(row["attractiveness_tier"], row["gap_interpretation"]),
        })

executive_matrix = pd.DataFrame(executive_rows)

display(style_table(
    executive_matrix,
    gradient_cols=["score"],
    gradient_cmap="RdYlGn",
    format_dict={"score": "{:.3f}", "rank": "{:.0f}"},
).set_caption("Matriz ejecutiva de implicaciones por línea"))


,business_line,country_iso3,rank,score,tier,gap,decision_implication
0,IB,USA,1,0.699,Alta,Diferencia moderada,Mantener en shortlist; diferenciar por drivers y restricciones
1,IB,CAN,2,0.681,Alta,Diferencia moderada,Mantener en shortlist; diferenciar por drivers y restricciones
2,IB,ESP,3,0.655,Alta,Diferencia moderada,Mantener en shortlist; diferenciar por drivers y restricciones
3,IB,CHL,4,0.638,Alta,Diferencia material,Priorizar análisis de entrada / business case
4,IB,URY,5,0.586,Alta,Diferencia débil,Mantener en shortlist; diferenciar por drivers y restricciones
5,IB,BHS,6,0.576,Media-alta,Diferencia moderada,Profundizar solo si existe encaje con línea específica
6,IB,BRA,7,0.554,Media-alta,Diferencia moderada,Profundizar solo si existe encaje con línea específica
7,IB,CRI,8,0.538,Media-alta,Diferencia débil,Profundizar solo si existe encaje con línea específica
8,IB,JAM,9,0.527,Media-alta,Empate práctico,Profundizar solo si existe encaje con línea específica
9,IB,PAN,10,0.526,Media-alta,Diferencia moderada,Profundizar solo si existe encaje con línea específica


In [411]:
scores_dir = resolve_data_path(configs["settings"]["data"].get("scores_path", "data/scores"))

# Componentes descompuestos
component_display.to_parquet(scores_dir / "component_decomposition_latest.parquet", index=False)

# TOPSIS by line en formato largo
topsis_long_rows = []
scores_dir = resolve_data_path(configs["settings"]["data"].get("scores_path", "data/scores"))
scores_dir.mkdir(parents=True, exist_ok=True)


for bl_key, bl_df in results["business_line_rankings"].items():
    tmp = ensure_country_column(bl_df)[["country_iso3", "score", "rank"]].copy()
scores_dir = resolve_data_path(configs["settings"]["data"].get("scores_path", "data/scores"))
scores_dir.mkdir(parents=True, exist_ok=True)


topsis_long_rows = []
for bl_key, bl_df in results["business_line_rankings"].items():
    tmp = ensure_country_column(bl_df)[["country_iso3", "score", "rank"]].copy()
    tmp.columns = ["country_iso3", "topsis_score", "topsis_rank"]
    tmp["business_line"] = bl_key
    topsis_long_rows.append(tmp)

topsis_long = pd.concat(topsis_long_rows, ignore_index=True)
topsis_long.to_parquet(scores_dir / "topsis_by_line_latest.parquet", index=False)

radar_by_line_df = results["radar_by_line"].copy()
radar_by_line_df.to_parquet(scores_dir / "radar_by_line_latest.parquet", index=False)

success_box(
    "Persistencia completada",
    f"Archivos guardados en {scores_dir}: component_decomposition_latest, "
    "topsis_by_line_latest, radar_by_line_latest."
)

In [412]:
results["wide_raw"]

variable,gdp_nominal,gdp_per_capita_ppp,gdp_growth,inflation_rate,population_total,urban_population_pct,unemployment_rate,current_account_gdp,public_debt_gdp,trade_openness,...,geographic_distance_km,common_language_spanish,hofstede_pdi,hofstede_idv,hofstede_mas,hofstede_uai,hofstede_lto,hofstede_ivr,colombian_diaspora_stock,cultural_distance_hofstede
country_iso3,,,,,,,,,,,,,,,,,,,,,
ARG,27.182177,30431.193122,-1.342931,219.883929,17.637525,92.274232,7.145,0.893118,68.944038,27.929761,...,8.457655,1.0,49.0,51.0,56.0,86.0,29.0,62.0,9.493336,43.335897
BHS,23.485350,41197.934255,3.378666,0.409162,12.902425,81.324926,9.207,-6.648608,71.457821,79.242459,...,7.779467,0.0,47.0,38.0,65.0,45.0,14.0,67.0,5.963579,45.022217
BLZ,21.887551,14346.518805,3.504664,3.289560,12.941017,41.753211,8.856,-1.612700,105.804729,108.972447,...,7.754053,0.0,80.0,19.0,40.0,86.0,22.5,89.0,3.637586,34.485504
BOL,24.728439,12877.635118,-1.123356,5.099766,16.334280,71.236145,2.967,-2.377848,68.944038,46.977872,...,7.501634,1.0,78.0,23.0,42.0,87.0,21.0,46.0,7.976939,47.791213
BRA,28.413013,22338.476564,3.419315,4.367464,19.172090,87.895855,5.970,-3.027160,81.855443,35.584524,...,8.368925,0.0,69.0,36.0,49.0,76.0,28.0,59.0,10.000206,36.796739
CAN,28.439119,64610.379517,1.554795,2.381584,17.536097,82.700257,6.907,-0.493602,64.887158,65.149921,...,8.571871,0.0,39.0,72.0,52.0,48.0,54.0,68.0,9.938662,79.561297
CHL,26.523168,36181.156617,2.644312,4.297639,16.799412,88.995091,8.974,-1.469331,68.944038,63.859857,...,8.354910,1.0,63.0,49.0,28.0,86.0,12.0,68.0,10.581166,44.821870
CRI,25.280825,31106.764356,4.321224,-0.412853,15.450599,79.310767,6.843,-0.946456,105.804729,71.331014,...,7.074117,1.0,35.0,15.0,21.0,86.0,22.5,89.0,9.142276,58.423026
DOM,25.545821,27541.662528,4.953522,3.302233,16.251538,72.412187,5.092,-3.353013,84.662111,51.783514,...,7.082549,1.0,65.0,38.0,65.0,45.0,11.0,54.0,10.320651,46.658333


In [413]:
fig = px.bar(
    results["wide_raw"]["gdp_nominal"].reset_index().sort_values("gdp_nominal"),
    x="gdp_nominal",
    y="country_iso3",
    orientation="h",
    color="gdp_nominal",
    color_continuous_scale=[[0, CIBEST["green"]], [0.5, CIBEST["gold"]], [1, CIBEST["red"]]],
    title="Países con mayor GDP nominal",
)
fig.update_layout(xaxis_title="GDP nominal", yaxis_title="País")
fig.show()

In [414]:
# Descubrir variables disponibles y mapearlas a dimensiones
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{len(available_vars)} variables disponibles en wide_raw:")

dim_vars_available: Dict[str, List[str]] = {}
for var in available_vars:
    meta = variable_catalog.get(var, {})
    dim = meta.get("dimension", "other")
    dim_vars_available.setdefault(dim, []).append(var)
    
for dim, vars_list in sorted(dim_vars_available.items()):
    print(f"  {dim}: {', '.join(vars_list)}")


47 variables disponibles en wide_raw:
  digital_tech: internet_users_pct, mobile_subscriptions, digital_payment_adults_pct, secure_internet_servers_per_million, commercial_bank_branches_per_100k_adults, atms_per_100k_adults, ict_goods_exports_pct_total_goods_exports
  financial: domestic_credit_private_gdp, account_ownership, interest_rate_spread, bank_npl_ratio, stock_market_cap_gdp, personal_remittances_gdp, bank_concentration_5, financial_system_deposits_gdp, bank_capital_rwa, return_on_equity
  institutional: regulatory_quality, government_effectiveness, rule_of_law, political_stability, voice_accountability, control_of_corruption, country_risk_premium, heritage_efi
  macro: gdp_nominal, gdp_per_capita_ppp, gdp_growth, inflation_rate, population_total, urban_population_pct, unemployment_rate, current_account_gdp, public_debt_gdp, trade_openness, fdi_net_inflows_gdp, weighted_mean_applied_tariff_all_products
  other: cultural_distance_hofstede
  proximity: geographic_distance_km, c

In [415]:
# Helper genérico para barplot horizontal de una variable
def barplot_variable(var_name: str, title: str, xlabel: str, reverse_color: bool = False):
    if var_name not in wide_raw.columns:
        display(Markdown(f"*Variable `{var_name}` no disponible en wide_raw.*"))
    return
    plot_df = results["wide_raw"][results["wide_raw"]["country_iso3"].isin(top_15)][["country_iso3", var_name]].dropna()
    plot_df = plot_df.sort_values(var_name, ascending=True)
    cscale = [[0, C["green"]], [0.5, C["gold"]], [1, C["red"]]] if reverse_color else [[0, C["gray_light"]], [1, C["gold"]]]
    fig = px.bar(plot_df, x=var_name, y="country_iso3", orientation="h",
                title=title, color=var_name, color_continuous_scale=cscale)
    fig.update_layout(xaxis_title=xlabel, yaxis_title="", height=480, font=dict(family="Arial"))
    fig.show()

In [416]:
for var in dim_vars_available.get("macro", []):
    print(var)
    barplot_variable(var, f"{var} por país", var, reverse_color=True)

gdp_nominal
gdp_per_capita_ppp
gdp_growth
inflation_rate
population_total
urban_population_pct
unemployment_rate
current_account_gdp
public_debt_gdp
trade_openness
fdi_net_inflows_gdp
weighted_mean_applied_tariff_all_products


In [417]:
display(Markdown("### D1 — Macroeconómica"))
for var in dim_vars_available.get("macro", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D1 — Macroeconómica


## 18. Limitaciones interpretativas que deben explicitarse

1. **TOPSIS es relativo al universo evaluado.** Si se excluyen o agregan países, cambian ideales positivos/negativos y pueden cambiar scores.
2. **Los pesos expresan tesis, no verdades universales.** Una línea puede tener una tesis bien estructurada y aun así correlacionar con otra si los países comparten fundamentos.
3. **Las contribuciones ponderadas no son una descomposición exacta del score TOPSIS.** Sirven para narrativa y perfilamiento; el efecto marginal ayuda a validar materialidad.
4. **IPC y Trend modulan el ranking, pero no sustituyen fundamentos estructurales.** Un país cercano no debe priorizarse si sus restricciones estructurales son severas.
5. **La imputación debe ser residual.** La exclusión por calidad debe ocurrir antes de imputar para no construir ideales con países artificialmente completados.



## 19. Conclusiones ejecutivas del Notebook 03

1. **RADAR global prioriza mercados que combinan fundamentos, cercanía y momentum.** No es equivalente a TOPSIS puro.
2. **Los países estructuralmente fuertes pueden bajar en RADAR si tienen menor afinidad con Colombia o bajo momentum reciente.** Esto no es un error: es parte de la lógica estratégica del score compuesto.
3. **Los rankings por línea reflejan tesis de negocio, no listas independientes.** La correlación alta entre líneas digitales puede ser válida cuando existe un factor común de madurez digital-institucional.
4. **La decisión debe comunicarse por bandas y gaps.** Diferencias marginales no deben convertirse en narrativas de superioridad estratégica.
5. **La explicabilidad requiere cruzar contribución y marginal.** Los drivers robustos son los que combinan peso, desempeño y efecto en ranking.
6. **El siguiente paso obligatorio es robustez probabilística.** La versión determinística debe validarse en Notebook 04 mediante Monte Carlo RADAR.

---

## Síntesis Ejecutiva

RADAR Cibest no entrega un ranking único; entrega un mapa de tesis de atractividad por país y línea. La lectura robusta combina ranking, banda, gap, componentes RADAR, correlación entre líneas, drivers, restricciones y sensibilidad a la tesis de negocio. La decisión ejecutiva debe concentrarse en mercados con alta atractividad, brecha material, drivers robustos y factibilidad estratégica para Cibest.
